# Univariate and Multivariate Analysis

This notebook uses Seaborn's `diamonds` dataset to demonstrate exploratory data analysis.

- **Univariate analysis:** `price` is analyzed by itself to understand its distribution, center, spread, skewness, and possible outliers.
- **Multivariate analysis:** `carat`, `price`, and `cut` are analyzed together to understand how diamond weight and cut quality relate to price.

## 1. Import Seaborn and load the diamonds dataset

Seaborn provides the built-in `diamonds` dataset and the statistical visualizations used later in the analysis.

In [ ]:
# Import Seaborn for loading the dataset and creating statistical charts.
import seaborn as sns

# Load Seaborn's diamonds dataset into a pandas DataFrame.
diamonds = sns.load_dataset("diamonds")

## 2. Import additional statistical libraries

NumPy supports numerical calculations, pandas supports tabular summaries, and Matplotlib provides figure-level control for the visualizations.

In [ ]:
# Import relevant libraries for numerical, tabular, and visual analysis.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Apply a consistent visual theme to all Seaborn charts.
sns.set_theme(style="whitegrid")

## 3. Inspect the dataset

Initial inspection confirms the dataset's dimensions, column types, sample records, and missing-value counts before analysis begins.

In [ ]:
# Display the number of rows and columns in the dataset.
print("Dataset shape:", diamonds.shape)

# Display the first five rows to review the available variables.
print(diamonds.head())

# Count missing values to confirm whether data cleaning is required.
print("\nMissing values by column:")
print(diamonds.isna().sum())

## 4. Univariate analysis of price

Univariate analysis examines one variable at a time. I selected `price` because it is a central numeric measure in the dataset. The statistics below describe its typical value, variability, quartiles, range, and skewness.

In [ ]:
# Select the price variable used for the univariate analysis.
price = diamonds["price"]

# Calculate descriptive statistics for price.
price_summary = price.describe()
price_statistics = pd.Series({
    "mean": np.mean(price),
    "median": np.median(price),
    "sample_variance": np.var(price, ddof=1),
    "sample_standard_deviation": np.std(price, ddof=1),
    "skewness": price.skew(),
})

print("Price descriptive summary:")
print(price_summary)
print("\nAdditional price statistics:")
print(price_statistics)

### Visualize the price distribution

The histogram and density curve show the shape of the distribution. The box plot makes the median, interquartile range, and unusually high price values easier to identify.

In [ ]:
# Create side-by-side charts for a complete univariate view of price.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot the frequency distribution and a smooth density estimate.
sns.histplot(price, bins=40, kde=True, color="royalblue", ax=axes[0])
axes[0].axvline(price.mean(), color="red", linestyle="--", label="Mean")
axes[0].axvline(price.median(), color="green", linestyle="--", label="Median")
axes[0].set_title("Distribution of Diamond Prices")
axes[0].set_xlabel("Price (US dollars)")
axes[0].legend()

# Plot price as a box plot to highlight spread and potential outliers.
sns.boxplot(x=price, color="skyblue", ax=axes[1])
axes[1].set_title("Box Plot of Diamond Prices")
axes[1].set_xlabel("Price (US dollars)")

plt.tight_layout()
plt.show()

### Univariate interpretation

The `price` distribution is right-skewed: most diamonds are concentrated in the lower price range, while a smaller number of expensive diamonds extend the upper tail. The mean is therefore higher than the median. The box plot also identifies many high-price observations beyond the upper whisker. These values are not automatically errors; they may represent legitimate premium diamonds.

## 5. Multivariate analysis of carat, price, and cut

Multivariate analysis examines relationships among multiple variables. I selected `carat` as the main numeric predictor, `price` as the numeric outcome, and `cut` as the categorical grouping variable.

In [ ]:
# Summarize carat and price for each cut category.
# observed=True reports only categories that are present in the data.
cut_summary = (
    diamonds.groupby("cut", observed=True)
    .agg(
        diamond_count=("price", "size"),
        average_carat=("carat", "mean"),
        median_price=("price", "median"),
        average_price=("price", "mean"),
    )
    .round(2)
)

print(cut_summary)

### Visualize carat, price, and cut together

The scatter plot uses position for `carat` and `price` and color for `cut`. A reproducible sample of 5,000 rows keeps the chart readable while retaining the overall pattern.

In [ ]:
# Take a reproducible sample to avoid overplotting all 53,940 diamonds.
diamond_sample = diamonds.sample(n=5000, random_state=42)

# Plot price against carat and use color to distinguish cut categories.
plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=diamond_sample,
    x="carat",
    y="price",
    hue="cut",
    palette="viridis",
    alpha=0.60,
    s=35,
)
plt.title("Diamond Price by Carat and Cut")
plt.xlabel("Carat")
plt.ylabel("Price (US dollars)")
plt.legend(title="Cut", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Examine correlations among all numeric variables

A correlation matrix provides another multivariate view. Values near 1 or -1 indicate strong linear relationships, while values near 0 indicate weak linear relationships.

In [ ]:
# Calculate Pearson correlations for every numeric column.
numeric_correlation = diamonds.select_dtypes(include=np.number).corr()

# Display the correlations as an annotated heatmap.
plt.figure(figsize=(10, 7))
sns.heatmap(
    numeric_correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
)
plt.title("Correlation Matrix for Numeric Diamond Variables")
plt.tight_layout()
plt.show()

# Print the carat-price correlation directly for easy interpretation.
carat_price_correlation = numeric_correlation.loc["carat", "price"]
print(f"Correlation between carat and price: {carat_price_correlation:.3f}")

### Multivariate interpretation

The scatter plot and correlation matrix show a strong positive relationship between `carat` and `price`: larger diamonds generally cost more. The colored groups show that diamonds from every cut category follow this upward pattern, although price ranges overlap among cut levels. This demonstrates why price should not be interpreted from cut alone—carat and other quality characteristics must also be considered.

## 6. Conclusion

The univariate analysis showed that diamond prices are widely dispersed and right-skewed. The multivariate analysis showed that carat has a strong positive relationship with price and that cut provides additional context to that relationship. Together, these methods help identify distribution patterns, unusual values, and relationships that can guide deeper statistical analysis or predictive modeling.